# Дообучение YandexGPT-5-Lite-8B-pretrain

Ноутбук подготавливает LoRA/SFT-обучение для задачи преобразования русских аналитических запросов в JSON-фильтры API.


## Что нужно до запуска

- GPU с CUDA и желательно 16+ GB VRAM но у меня 8 GB и работает
- `HF_TOKEN` в корневом `.env`
- датасет уже лежит в `training/llm_sft/`


In [1]:
!pip install -q -U datasets transformers accelerate peft trl bitsandbytes sentencepiece huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.5 MB/s eta 0:00:00


In [7]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import torch
from datasets import Features, Value, load_dataset
from huggingface_hub import login
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

In [8]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_DIR = DRIVE_ROOT if DRIVE_ROOT.exists() else Path.cwd()

print('IN_COLAB:', IN_COLAB)
print('PROJECT_DIR:', PROJECT_DIR)

Mounted at /content/drive
IN_COLAB: True
PROJECT_DIR: /content/drive/MyDrive


In [9]:
def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

try:
    from google.colab import userdata
    secret_token = userdata.get('HF_TOKEN')
    if secret_token:
        os.environ.setdefault('HF_TOKEN', secret_token)
except Exception:
    pass

load_env_file(PROJECT_DIR / '.env')
load_env_file(Path('/content/.env'))

hf_token = os.getenv('HF_TOKEN')
if not hf_token:
    raise RuntimeError(
        'HF_TOKEN не найден. Добавь его в Colab Secrets как HF_TOKEN '
        'или загрузи .env с HF_TOKEN=...'
    )

login(token=hf_token, add_to_git_credential=False)
print('HF token loaded successfully')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF token loaded successfully


In [10]:
MODEL_NAME = 'yandex/YandexGPT-5-Lite-8B-pretrain'

TRAIN_PATH = PROJECT_DIR / 'training' / 'llm_sft' / 'budget_query_sft_train.jsonl'
VAL_PATH = PROJECT_DIR / 'training' / 'llm_sft' / 'budget_query_sft_val.jsonl'

DATASET_CACHE_DIR = PROJECT_DIR / '.cache' / 'hf_datasets'
OUTPUT_DIR = PROJECT_DIR / 'artifacts' / 'yandexgpt5-lite-budget-query-lora'

DATASET_CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SEQ_LENGTH = 1536

print('TRAIN_PATH:', TRAIN_PATH, TRAIN_PATH.exists())
print('VAL_PATH:', VAL_PATH, VAL_PATH.exists())
print('DATASET_CACHE_DIR:', DATASET_CACHE_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

TRAIN_PATH: /content/drive/MyDrive/training/llm_sft/budget_query_sft_train.jsonl True
VAL_PATH: /content/drive/MyDrive/training/llm_sft/budget_query_sft_val.jsonl True
DATASET_CACHE_DIR: /content/drive/MyDrive/.cache/hf_datasets
OUTPUT_DIR: /content/drive/MyDrive/artifacts/yandexgpt5-lite-budget-query-lora


In [11]:
if IN_COLAB and (not TRAIN_PATH.exists() or not VAL_PATH.exists()):
    from google.colab import files

    print('Файлы датасета не найдены в ожидаемом месте.')
    print('Загрузи budget_query_sft_train.jsonl и budget_query_sft_val.jsonl.')
    uploaded = files.upload()

    local_data_dir = Path('/content/training/llm_sft')
    local_data_dir.mkdir(parents=True, exist_ok=True)

    for name, content in uploaded.items():
        (local_data_dir / name).write_bytes(content)

    TRAIN_PATH = local_data_dir / 'budget_query_sft_train.jsonl'
    VAL_PATH = local_data_dir / 'budget_query_sft_val.jsonl'

print('TRAIN_PATH:', TRAIN_PATH, TRAIN_PATH.exists())
print('VAL_PATH:', VAL_PATH, VAL_PATH.exists())

if not TRAIN_PATH.exists() or not VAL_PATH.exists():
    raise FileNotFoundError('Не найдены train/val jsonl-файлы. Проверь пути выше.')

TRAIN_PATH: /content/drive/MyDrive/training/llm_sft/budget_query_sft_train.jsonl True
VAL_PATH: /content/drive/MyDrive/training/llm_sft/budget_query_sft_val.jsonl True


In [12]:
def load_budget_query_sft_dataset(train_path: Path, val_path: Path, cache_dir: Path):
    features = Features({
        'text_query': Value('string'),
        'prompt': Value('string'),
        'completion': Value('string'),
    })

    dataset = load_dataset(
        'json',
        data_files={
            'train': str(train_path),
            'validation': str(val_path),
        },
        features=features,
        cache_dir=str(cache_dir),
    )

    def add_text(row):
        prompt = row.get('prompt') or ''
        completion = row.get('completion') or ''
        row['text'] = prompt + completion
        return row

    return dataset.map(add_text)

dataset = load_budget_query_sft_dataset(TRAIN_PATH, VAL_PATH, DATASET_CACHE_DIR)
dataset

Generating train split: 0 examples [00:00, ? examples/s]

DatasetGenerationError: An error occurred while generating the dataset

In [ ]:
sample = dataset['train'][0]
print(sample['text_query'])
print(sample['prompt'][:700])
print(sample['completion'])

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Для дообучения нужен GPU с CUDA. В Colab включи Runtime → Change runtime type → GPU.')

major, minor = torch.cuda.get_device_capability()
supports_bf16 = major >= 8
print('GPU:', torch.cuda.get_device_name(0))
print('bf16 supported:', supports_bf16)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if supports_bf16 else torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token, legacy=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=hf_token,
    device_map='auto',
    torch_dtype=torch.bfloat16 if supports_bf16 else torch.float16,
    quantization_config=bnb_config,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

GPU: Tesla T4
bf16 supported: False


config.json:   0%|          | 0.00/542 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/2.57M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules='all-linear',
)

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field='text',
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    bf16=supports_bf16,
    fp16=not supports_bf16,
    gradient_checkpointing=True,
    report_to='none',
    optim='paged_adamw_8bit',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()

In [ ]:
trainer.train()

In [ ]:
adapter_dir = OUTPUT_DIR / 'adapter'
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print('Saved adapter to:', adapter_dir)

In [ ]:
test_queries = [
    'Покажи лимиты по Благовещенску по месяцам',
    'Покажи кассовые выплаты по 0502',
    'Покажи сумму контрактов по источнику gz',
]

def build_inference_prompt(query: str) -> str:
    return (
        'Ты преобразуешь русский запрос к системе бюджетной аналитики в JSON для API. '
        'Верни только JSON без пояснений.\n\n'
        f'Запрос пользователя:\n{query}\n\nВерни только JSON:'
    )

model.eval()
for query in test_queries:
    prompt = build_inference_prompt(query)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=220,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print('QUERY:', query)
    print(generated)
    print('-' * 80)